## Libraries needed for calculation

In [1]:
import numpy as np
from openmm import *
from openmm.app import *
from openmm.unit import *
import random
from mdtraj.reporters import HDF5Reporter
from scipy.optimize import curve_fit
from scipy.interpolate import CubicSpline

## Some physical constants and coefficients to convert external units in openmm internal units

In [2]:
E_CHARGE = 1.602176634e-19
AVOGADRO_CONSTANT = 6.02214076e23
AEM = 1.66053906892e-27
e_field_convert = 1.0 / (
    1e12 / AVOGADRO_CONSTANT / E_CHARGE
)  # multiply eV/m by this to get internal units
b_field_convert = 1e-12 * E_CHARGE / AEM  # multiply tesla by this to get internal units
cm_to_nm = 1e7

## SNF mass distribution

In [3]:
# parameters for distribution of heavy and light fission products
m_light = 108
m_heavy = 207

masses = np.array([m_light, m_heavy])
n_ions = np.array([250, 250])

"""
# checking distribution of ions which will be used in simulation
plt.figure()
plt.scatter(masses, n_ions)
print(masses)
print(np.sum(n_ions))
"""

'\n# checking distribution of ions which will be used in simulation\nplt.figure()\nplt.scatter(masses, n_ions)\nprint(masses)\nprint(np.sum(n_ions))\n'

## Adjustable parameters

In [4]:
injector_pos = 18  # in cm
injector_pos *= cm_to_nm
E_field = 2000  # in volt / meter
time_step = 0.2e6  # timestep in picoseconds


# Electric field fluctuation parameters
rel = 0.1
amplitude = E_field / 100 * 25 * rel  # in V
frequency = 20e3  # Hz
frequency_convert = 1e12  # 1/seconds to 1/picoseconds


box_size = 5  # meter

# initial velocities and coordinates
velocities = []
positions = []
current_ion_number = 0
for i in range(n_ions.size):
    for k in range(int(n_ions[i])):
        # Ez = random.uniform(1,20) # z component of energy in eV
        Ez = 5  # z component of energy in eV
        vz = (2.0 * Ez / masses[i] * E_CHARGE / AEM) ** 0.5 / 1e3
        vx = 0
        vy = 0
        velocities.append([vx, vy, vz])
        positions.append([injector_pos, 0, 0])
        current_ion_number += 1

## Conversion of some internal variables

In [5]:
E_field *= e_field_convert
injector_pos *= meter.conversion_factor_to(nanometer) / 100
box_size *= meter.conversion_factor_to(nanometer)

## System (topology) creation


In [6]:
system = System()
box_matrix = box_size * np.identity(3)

system.setDefaultPeriodicBoxVectors(*box_matrix)

for i in range(n_ions.size):
    for _iatom in range(int(n_ions[i])):
        system.addParticle(masses[i])

# Define a relatively boring topology object.
topology = Topology()
topology.setPeriodicBoxVectors(box_matrix)
chain = topology.addChain()
residue = topology.addResidue("ions", chain)
for i in range(n_ions.size):
    element = Element.getByMass(masses[i])
    for _iatom in range(int(n_ions[i])):
        topology.addAtom(masses[i], element, residue)

## Magnetic field creation

In [7]:
# Set parametrs of regular cubic grid on which magnetic field was calculated
rsize = 173  # number of points for r
rmin = 0  # minimum r value in cm
rmax = 43  # # maximum r value in cm

zsize = 403  # number of points for z
zmin = -0.5  # maximum z value in cm
zmax = 100  # minimum z value in cm

# Loading field from file
B = np.loadtxt("mfco_4_coils.csv", skiprows=9, delimiter=",")
B_field_x = B[:, 2]
B_field_y = B[:, 3]
B_field_z = B[:, 4]

# Creating openmm tabulated funtions for magnetic field
magnetic_field_function_x = Continuous2DFunction(
    rsize,
    zsize,
    B_field_x * b_field_convert,
    rmin * cm_to_nm,
    rmax * cm_to_nm,
    zmin * cm_to_nm,
    zmax * cm_to_nm,
)
magnetic_field_function_y = Continuous2DFunction(
    rsize,
    zsize,
    B_field_y * b_field_convert,
    rmin * cm_to_nm,
    rmax * cm_to_nm,
    zmin * cm_to_nm,
    zmax * cm_to_nm,
)
magnetic_field_function_z = Continuous2DFunction(
    rsize,
    zsize,
    B_field_z * b_field_convert,
    rmin * cm_to_nm,
    rmax * cm_to_nm,
    zmin * cm_to_nm,
    zmax * cm_to_nm,
)

xsize = ysize = phasesize = 50
xmin = ymin = -25  # in cm
xmax = ymax = 25  # in cm
phasemin = 0
phasemax = 2 * np.pi

data = np.load("fluctuation_x_y_phase.npz")
E_field_x = data["Ex"][::125]
E_field_y = data["Ey"][::125]


E_field_x *= amplitude
E_field_y *= amplitude

electric_field_fluctuation_function_x = Continuous3DFunction(
    xsize,
    ysize,
    phasesize,
    E_field_x * e_field_convert,
    xmin * cm_to_nm,
    xmax * cm_to_nm,
    ymin * cm_to_nm,
    ymax * cm_to_nm,
    phasemin,
    phasemax,
)

electric_field_fluctuation_function_y = Continuous3DFunction(
    xsize,
    ysize,
    phasesize,
    E_field_y * e_field_convert,
    xmin * cm_to_nm,
    xmax * cm_to_nm,
    ymin * cm_to_nm,
    ymax * cm_to_nm,
    phasemin,
    phasemax,
)

## Creating Boris integrator

In [8]:
# Creating integrator
integrator = CustomIntegrator(time_step)
## Adding tabulated functions defined earlier
integrator.addTabulatedFunction("Bx", magnetic_field_function_x)
integrator.addTabulatedFunction("By", magnetic_field_function_y)
integrator.addTabulatedFunction("Bz", magnetic_field_function_z)
integrator.addTabulatedFunction(
    "E_fluctuation_x", electric_field_fluctuation_function_x
)
integrator.addTabulatedFunction(
    "E_fluctuation_y", electric_field_fluctuation_function_y
)

## Creating variables needed for calulation, including E and B fields
integrator.addPerDofVariable("v_plus", 0)
integrator.addPerDofVariable("v_minus", 0)
integrator.addPerDofVariable("v_prime", 0)
integrator.addUpdateContextState()
integrator.addPerDofVariable("E", 0)
integrator.addPerDofVariable("B", 0)
integrator.addPerDofVariable("t", 0)
integrator.addPerDofVariable("r_2d", 0)

# for E fluctuation.
integrator.addPerDofVariable("phase", 0)
integrator.beginIfBlock("time = 0")
integrator.addComputePerDof("phase", "uniform")
integrator.endBlock()
integrator.beginIfBlock("time != 0")
integrator.addComputePerDof(
    "phase", "phase + dt * {0}".format(frequency * frequency_convert)
)
integrator.endBlock()

## Defining integrator performance during one step
### Calculating E and B fields
integrator.addComputePerDof("r_2d", "vector(0,0,0)-vector(_x(x),_y(x),0)")
integrator.addComputePerDof(
    "B",
    "vector(Bx(sqrt(dot(r_2d,r_2d)), _z(x)), By(sqrt(dot(r_2d,r_2d)), _z(x)), Bz(sqrt(dot(r_2d,r_2d)), _z(x)))",
)
integrator.addComputePerDof(
    "E",
    "r_2d*vector({0},{0},1) / sqrt(dot(r_2d,r_2d))".format(E_field * e_field_convert),
)  # linear potential
integrator.addComputePerDof(
    "E",
    "E + vector(E_fluctuation_x(_x(x), _y(x), _x(phase)), E_fluctuation_y(_x(x), _y(x), _x(phase)),0)",
)  # E fluctuation

integrator.addComputePerDof("t", "0.5/m*B*dt")

### Changing velocity and position
integrator.addComputePerDof("v_minus", "v+1/m*E*dt*0.5")
integrator.addComputePerDof("v_prime", "v_minus+cross(v_minus,t)")
integrator.addComputePerDof("v_plus", "v_minus+cross(v_prime,2/(1+dot(t,t))*t)")
integrator.addComputePerDof("v", "v_plus+0.5/m*E*dt")
integrator.addComputePerDof("x", "x+v*dt")

16

## Creating simulation

In [9]:
simulation = Simulation(topology, system, integrator)
# Assigning initial positions and velocities to ions
simulation.context.setVelocities(velocities)
simulation.context.setPositions(positions)

# Add reporters, which will write some outputs to file
simulation.reporters = []

## Reporters for ovito visualization

### Writing initial pdb file with topology
positions_0 = simulation.context.getState(getPositions=True).getPositions()
with open("init.pdb", "w") as f:
    PDBxFile.writeFile(topology, positions_0, f)

dcd_reporter = DCDReporter("output.dcd", 5)
simulation.reporters.append(dcd_reporter)

## Reporter for future postprocessing in Separation repository
hdf5_reporter = HDF5Reporter(
    "ion_trajectories_openmm.h5",
    5,
    velocities=True,
    potentialEnergy=False,
    kineticEnergy=False,
    temperature=False,
)
simulation.reporters.append(hdf5_reporter)

## Running simulation

In [10]:
# Manually writing 0-th step to h5 traectory file, as it is not written automatically
hdf5_reporter.report(
    simulation,
    simulation.context.getState(
        getPositions=True, getVelocities=True, getParameters=True
    ),
)

n_steps = 450  # number of simulation steps
simulation.step(n_steps)  # perform simulation

hdf5_reporter.close()  # close file with trajectory

In [11]:
print(current_ion_number)

500
